<a href="https://colab.research.google.com/github/antonyza/multimodal-music-analysis/blob/main/part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Filtering
Keep the songs belonging to 5 genres only.

Those genres are personally picked.

In [33]:
import pandas as pd

url = 'https://raw.githubusercontent.com/antonyza/multimodal-music-analysis/refs/heads/main/Data/id_genres.csv'
my_genres = ['techno', 'country', 'emo', 'reggae', 'jazz']
chunk_size = 500
filtered_id_genres = []

for chunk in pd.read_csv(url, sep='\t', chunksize=chunk_size):
    chunk = chunk.dropna(subset=['genres']).copy() # Remove empty lines
    chunk['genres'] = chunk['genres'].str.split(',') # Split genres column (strings) into a list of genres
    chunk = chunk.explode('genres') # Explode the list of genres into separate rows (same song ID)
    chunk = chunk[chunk['genres'].isin(my_genres)].rename(columns={"genres": "genre"}) # Filter the genres

    if not chunk.empty:
        filtered_id_genres.append(chunk)

df_genre = pd.concat(filtered_id_genres, ignore_index=True)
df_genre = df_genre.groupby('id')['genre'].apply(
    lambda x: list(x.unique()) if len(x.unique()) > 1 else x.iloc[0] # remove duplicates
).reset_index()

df_genre.head(5)

,id,genre
0,00KSCJkYb8JKa4Y3,emo
1,0130k7BE0WlWUQLg,emo
2,02RGE9FNH65RtMS7,techno
3,02S1suSkE5dLFuQh,techno
4,02cqDSNdfTyOD1AI,country


Filter the other CSV file data based on the song IDs we kept

In [12]:
valid_ids = set(df_genre['id'])

# ID-Information
url = 'https://raw.githubusercontent.com/antonyza/multimodal-music-analysis/refs/heads/main/Data/id_information.csv'
filtered_id_info = []

for chunk in pd.read_csv(url, sep='\t', chunksize=chunk_size):
    chunk = chunk[chunk['id'].isin(valid_ids)] # keep from valid song IDs
    if not chunk.empty:
        filtered_id_info.append(chunk)

df_id_info = pd.concat(filtered_id_info, ignore_index=True)


# ID-Tags
url = 'https://raw.githubusercontent.com/antonyza/multimodal-music-analysis/refs/heads/main/Data/id_tags.csv'
filtered_id_tags = []

for chunk in pd.read_csv(url, sep='\t', chunksize=chunk_size):
    chunk = chunk[chunk['id'].isin(valid_ids)] # keep from valid song IDs

    if not chunk.empty:
        filtered_id_tags.append(chunk)

df_id_tags = pd.concat(filtered_id_tags, ignore_index=True)

Get lyrics from filtered IDs

In [36]:
from os.path import isfile
import tarfile
import urllib.request

# Lyrics
url = 'https://raw.githubusercontent.com/antonyza/multimodal-music-analysis/main/Data/processed_lyrics.tar.gz'
tar_path = 'processed_lyrics.tar.gz'
urllib.request.urlretrieve(url, tar_path)
lyrics = []

with tarfile.open(tar_path, 'r:gz') as tar:
    for member in tar.getmembers():
        if member.isfile:
            file_id = member.name.split('/')[-1].split('.')[0] # Get ID from file name (remove .txt)

            if file_id in valid_ids:
                file = tar.extractfile(member)
                if file is not None:
                    content = file.read().decode('utf-8')
                    if not content.strip(): # if empty file (no lyrics) then its instrumental
                        content = 'instrumental'

                    lyrics.append({'id': file_id, 'lyrics': content})

df_lyrics = pd.DataFrame(lyrics)
df_lyrics.head()

,id,lyrics
0,YWEVpqA38ezG0dPU,citi shoe clueless blue pay view noman news bl...
1,wzU0Ts9ZAMfwPqAl,get crush sweeti pie day nighttim hear sigh ne...
2,aB5MGbb1KGWRgl2N,tear open packag empti box mean empti box send...
3,M43AP50YsKZVwzbc,could ever let go go beg drive mad say someth ...
4,9Y3mWcWntTVnpXXh,levit round go stop know float away bend break...


Get MFCC stats

In [14]:
url = 'https://raw.githubusercontent.com/antonyza/multimodal-music-analysis/main/Data/id_mfcc_stats.tsv.bz2'
filtered_mfcc_stats = []

for chunk in pd.read_csv(url, sep='\t', chunksize=chunk_size):
    chunk = chunk[chunk['id'].isin(valid_ids)] # keep from valid song IDs

    if not chunk.empty:
        filtered_mfcc_stats.append(chunk)

df_mfcc_stats = pd.concat(filtered_mfcc_stats, ignore_index=True)
df_mfcc_stats.head()

,id,MFCC000,MFCC001,MFCC002,MFCC003,MFCC004,MFCC005,MFCC006,MFCC007,MFCC008,...,cov_9_9,cov_9_10,cov_9_11,cov_9_12,cov_10_10,cov_10_11,cov_10_12,cov_11_11,cov_11_12,cov_12_12
0,rLfjv8ackxbMBhNm,19.673954,7.374847,-15.109837,-11.847433,0.264044,-18.521660,-7.466475,-2.242330,-13.572355,...,118.288409,5.729727,-4.038114,-14.874527,114.725826,18.788491,-3.406951,97.033048,19.932110,91.360570
1,RLHbq9UXd6cof8Hs,24.639158,-12.822001,-7.392960,-6.209181,-2.153774,-2.274888,-3.867082,-1.193819,-1.405178,...,119.159003,54.040871,33.615619,22.343001,71.530292,38.250632,21.195860,84.002375,47.080897,77.398648
2,j1dQ6TZ3bXn6Ynyt,24.088564,-4.647918,-7.447447,2.401356,-10.061542,1.907399,-5.167847,3.960707,0.971683,...,89.620570,13.797518,8.197030,-7.193006,103.671687,10.435869,-0.186379,83.345863,1.543711,79.169231
3,lRAC2JboOOofsTuE,23.276661,-8.367137,-6.481616,-3.801927,-8.432757,-0.617347,-3.797227,2.791821,1.650162,...,116.750916,29.797668,2.512981,16.170507,103.003531,36.900034,11.820328,87.713964,27.236822,82.561620
4,lrbEv0Tu2jLNlMmD,23.848660,-17.799595,-11.282082,1.395223,-4.768649,1.460606,-0.365338,12.974216,1.694667,...,86.538263,33.140836,5.496330,-0.937074,94.077416,35.472765,-6.621396,72.174649,22.249008,77.636431


Merge and clean the above dataframes into one

In [38]:
# Merge
temp = pd.merge(df_genre, df_lyrics, on='id', how='inner')
df_merged = pd.merge(temp, df_mfcc_stats, on='id', how='inner')


# Clean
mfcc_cols = [col for col in df_merged.columns if col not in ['id', 'genre', 'lyrics']] # Get all MFCC columns
df_merged['mfcc_stats'] = df_merged[mfcc_cols].values.tolist() # Merge MFCC columns into one column (list)
df_merged = df_merged[['id', 'genre', 'lyrics', 'mfcc_stats']] # keep those 4 columns now

print('Data collection complete with final dataframe shape', df_merged.shape)
df_merged.head(20)

Data collection complete with final dataframe shape (7900, 4)


,id,genre,lyrics,mfcc_stats
0,00KSCJkYb8JKa4Y3,emo,fuck nobodi anybodi somebodi nobodi anybodi on...,"[24.40309333801269, -16.733760833740234, -8.50..."
1,0130k7BE0WlWUQLg,emo,tire wake tire well trite say tri chang leg ac...,"[23.711875915527344, -12.89763641357422, -5.54..."
2,02RGE9FNH65RtMS7,techno,fame hold vain letter play game hard love get ...,"[21.82469749450684, -18.16270637512207, -3.055..."
3,02S1suSkE5dLFuQh,techno,instrumental,"[23.05133056640625, -8.015624046325684, -28.29..."
4,02cqDSNdfTyOD1AI,country,hope day come easi moment pass slow road lead ...,"[24.18877601623535, -10.060083389282228, -10.9..."
5,03PbpZwYbPjAUBqB,jazz,go run go run go run go run oh sinn man go run...,"[23.98193550109864, -3.5930917263031006, -5.61..."
6,03dA0mAJj48nT2ab,jazz,tire day get home strang unfamiliar feel kind ...,"[23.773550033569336, -4.35776948928833, 0.2221..."
7,04KIChLwxgxSGbkD,jazz,instrumental,"[21.67498588562012, -3.3346436023712163, -5.62..."
8,04La8lPgQyShB5dr,jazz,want slave want work day want true want make l...,"[21.761722564697266, -16.000164031982422, -12...."
9,04SlsdFaQIDnfvr9,jazz,instrumental,"[23.14767074584961, -17.720975875854492, 2.716..."


In [26]:
from functools import reduce
import numpy as np

# merge
dataframes_to_merge = [df_genre, df_lyrics, df_mfcc_stats]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='id', how='inner'), dataframes_to_merge)

# clean
df_merged['lyrics'] = df_merged['lyrics'].replace(r'^\s*$', np.nan, regex=True) # clean empty lyrics
df_merged = df_merged.dropna(subset=['lyrics'])

mfcc_cols = [col for col in df_merged.columns if col not in ['id', 'genre', 'lyrics']] # merge mfcc columns into one column (list)
df_merged['mfcc_stats'] = df_merged[mfcc_cols].values.tolist()

# keep those final columns
df_merged = df_merged[['id', 'genre', 'lyrics', 'mfcc_stats']]

print(f"Data Collection Complete! Final dataset size: {len(df_merged)}")
df_merged.head(10)

Data Collection Complete! Final dataset size: 7290


,id,genre,lyrics,mfcc_stats
0,00KSCJkYb8JKa4Y3,emo,fuck nobodi anybodi somebodi nobodi anybodi on...,"[24.40309333801269, -16.733760833740234, -8.50..."
1,0130k7BE0WlWUQLg,emo,tire wake tire well trite say tri chang leg ac...,"[23.711875915527344, -12.89763641357422, -5.54..."
2,02RGE9FNH65RtMS7,techno,fame hold vain letter play game hard love get ...,"[21.82469749450684, -18.16270637512207, -3.055..."
5,02cqDSNdfTyOD1AI,country,hope day come easi moment pass slow road lead ...,"[24.18877601623535, -10.060083389282228, -10.9..."
6,02cqDSNdfTyOD1AI,country,hope day come easi moment pass slow road lead ...,"[24.18877601623535, -10.060083389282228, -10.9..."
7,03PbpZwYbPjAUBqB,jazz,go run go run go run go run oh sinn man go run...,"[23.98193550109864, -3.5930917263031006, -5.61..."
8,03dA0mAJj48nT2ab,jazz,tire day get home strang unfamiliar feel kind ...,"[23.773550033569336, -4.35776948928833, 0.2221..."
11,04La8lPgQyShB5dr,jazz,want slave want work day want true want make l...,"[21.761722564697266, -16.000164031982422, -12...."
13,04deZCVGi5AOgYIS,emo,young boy father take citi see march band say ...,"[21.852802276611328, -17.6103572845459, -11.83..."
14,04xUDjAYC14jsHyH,emo,talk feel crowd errand world save -pron- rain ...,"[24.16695022583008, -14.148749351501465, -12.8..."
